# 05. Runnable 기본 조합 (LCEL)

- LangChain에서의 Runnable은 입력값을 받아 특정 작업을 수행하고 출력값을 반환하는 실행 가능한 구성요소임.
- 이를 활용하면 프롬프트, 모델, 출력 파서, 사용자 정의 함수, 병렬 처리 흐름을 직관적인 파이프라인으로 표현할 수 있음.
- LCEL(LangChain Expression Language)은 `|` 연산자로 컴포넌트를 체인처럼 연결함.
- Runnable 조합을 사람이 읽기 쉬운 파이프라인 문법으로 표현한 것.

```
prompt | llm | parser
```

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. 기본 체인: prompt | llm | parser

In [2]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}에 대해 한 문장으로 설명해줘")
])

parser = StrOutputParser()

chain = prompt | llm | parser

result = chain.invoke({"topic": "벡터 DB"})
print(result)


벡터 DB는 고차원 벡터 데이터를 저장하고 검색하는 데 최적화된 데이터베이스로, 주로 머신러닝 및 인공지능 응용에서 유사성 검색에 사용됩니다.


## 2. 체인 스트리밍
- LangChain 체인 실행 결과를 스트리밍 방식으로 조금씩 받아 출력하는 코드

In [3]:


for chunk in chain.stream({"topic": "임베딩"}):
    # 현재 받은 응답 조각을 출력 후 줄바꿈 없이, 버퍼에 쌓아두지 않고 즉시 출력
    print(chunk, end="", flush=True)

임베딩(embedding)은 고차원 데이터를 저차원 공간에 매핑하여 의미 있는 벡터 표현으로 변환하는 기법으로, 주로 자연어 처리와 추천 시스템 등에서 사용됩니다.

In [4]:

for chunk in chain.stream({"topic": "임베딩"}):
    # Python이 출력 내용을 잠시 버퍼에 모아두었다가 나중에 보여줌
    print(chunk, end="", flush=False)


임베딩은 고차원 데이터를 저차원 공간에 매핑하여 의미 있는 벡터 표현으로 변환하는 기법입니다.

# 3. Runnable의 공통 실행 메서드
- 입력 값을 받아 어떤 작업을 수행하고, 출력값을 반환하는 표준 실행 단위
- 프롬프트, LLM, 출력파서, Retriever, 사용자 정의 함수 등을 하나의 실행 규칙으로 연결하기 위한 표준 인터페이스
- 기본 : 입력 → Runnable → 출력
- 여러 Runnable 연결 : 입력 → prompt → llm → parser → custom function → 최종 출력 

- Runnable의 공통 실행 메서드 

| 메서드                    | 의미              | 사용 상황           |
| ---------------------- | --------------- | --------------- |
| `invoke()`             | 단일 입력 실행        | 가장 기본 실행        |
| `ainvoke()`            | 비동기 단일 실행       | async 환경        |
| `batch()`              | 여러 입력을 한 번에 실행  | 여러 질문 일괄 처리     |
| `abatch()`             | 비동기 batch 실행    | 대량 처리           |
| `stream()`             | 결과를 조각 단위로 스트리밍 | 챗봇 실시간 출력       |
| `astream()`            | 비동기 스트리밍        | 웹소켓, SSE 등      |
| `batch_as_completed()` | 완료되는 순서대로 결과 반환 | 입력별 처리 시간이 다를 때 |


## 3. RunnablePassthrough - 입력을 그대로 전달
- RunnablePassthrough는 입력값을 거의 그대로 통과시킴

In [6]:
from langchain_core.runnables import RunnablePassthrough

# 입력값을 그대로 흘려보내기
chain=RunnablePassthrough()
result = chain.invoke({"topic":"aaa"})
print(result)

{'topic': 'aaa'}


- `.assign()`은 기존 딕셔너리에 새로운 필드를 추가할 때 사용함

In [8]:
# 실용 예: 입력을 그대로 유지하면서 추가 키 삽입
# 입력값에 "word" 키가 있다고 가정할 때, "upper" 키에 대문자 변환값을 추가하는 체인

from langchain_core.runnables import RunnablePassthrough

upper = lambda x:x["word"].upper()
chain = RunnablePassthrough.assign(upper=upper)

result = chain.invoke({"word":"aaaaaaaaaaa"})
print(result)


{'word': 'aaaaaaaaaaa', 'upper': 'AAAAAAAAAAA'}


## 4. RunnableLambda - 함수를 Runnable로 감싸기
- RunnableLambda는 일반 파이썬 함수를 Runnable로 감싸는 객체
- RunnableLambda는 체인 중간이나 마지막에 후처리 로직을 넣을 때 주로 사용

In [11]:
from langchain_core.runnables import RunnableLambda

# 문자열 -> 문자열 편집 반환 함수
def add_prefix(text: str) -> str:
    return f"[결과] {text}"

prompt = ChatPromptTemplate.from_messages([
    ("human", "{apple}에 대해 한 문장으로 설명해줘")
])

chain = prompt | llm | StrOutputParser() | RunnableLambda(add_prefix)

result = chain.invoke({"apple": "바나나"})
print(result)


[결과] 바나나는 달콤하고 부드러운 식감의 열대 과일로, 비타민과 미네랄이 풍부하여 건강에 유익합니다.


## 5. 병렬 실행 - RunnableParallel

딕셔너리 형태로 여러 체인을 동시에 실행합니다.

In [12]:
from langchain_core.runnables import RunnableParallel

prompt_ko = ChatPromptTemplate.from_messages([("human", "{topic}을 한국어로 설명해줘")])
prompt_en = ChatPromptTemplate.from_messages([("human", "Explain {topic} in English")])

chain_ko = prompt_ko | llm | StrOutputParser()
chain_en = prompt_en | llm | StrOutputParser()

parallel_chain = RunnableParallel(ko=chain_ko, en=chain_en)

result = parallel_chain.invoke({"topic": "포유류"})
print("[한국어 설명]", result["ko"])




[한국어 설명] 포유류(哺乳類)는 태반을 통해 태아를 발달시키고, 젖샘을 통해 새끼에게 젖을 먹여 기르는 동물의 한 분류입니다. 포유류는 일반적으로 체온을 일정하게 유지하는 온혈동물이며, 털이나 모피로 덮여 있는 경우가 많습니다. 이들은 다양한 서식지에서 발견되며, 육상, 수중, 공중 등 다양한 환경에 적응해 살아갑니다.

포유류는 크게 두 가지 주요 그룹으로 나눌 수 있습니다: 유태류(유대류)와 태반류. 유태류는 태반 없이 새끼를 낳고, 태반류는 태반을 통해 새끼를 임신하고 출산합니다. 포유류의 예로는 인간, 개, 고양이, 코끼리, 돌고래 등이 있습니다. 이들은 지능이 높고 사회적 행동을 보이는 경우가 많아, 생태계에서 중요한 역할을 합니다.


## 실습 문제
Q2. RunnableLambda를 활용해 LLM 응답을 받아 단어 수를 세는 함수를 파이프라인 끝에 추가해보세요.   결과로 {'text': '...', 'word_count': 42} 형태로 반환하세요.

In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# 1. 프롬프트 구성
prompt = ChatPromptTemplate.from_template(
    "{topic}에 대해 전문가수준으로 설명해줘."
)

# 2. LLM 생성
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# 3. 출력 파서 생성
parser = StrOutputParser()

# 4. LLM 응답을 받아 단어 수를 세는 함수
def count_words(text: str) -> dict:
    return{
        "text": text,
        "word_count": len(text.split())
    }


# 5. RunnableLambda로 함수 감싸기
word_counter = RunnableLambda(count_words)


# 6. LCEL 체인 구성
chain = prompt | llm | parser | word_counter

# 7. 실행
result = chain.invoke({"topic": "LangChain Runnable"})

print(result["word_count"], "words")
print("-"*10)
print(result["text"])

315 words
----------
LangChain의 `Runnable`은 LangChain 프레임워크 내에서 데이터 처리 및 작업 흐름을 구성하는 데 사용되는 핵심 개념 중 하나입니다. `Runnable`은 다양한 작업을 수행할 수 있는 객체로, 입력을 받아 처리하고 결과를 반환하는 기능을 제공합니다. 이 개념은 특히 자연어 처리(NLP) 및 AI 모델과의 상호작용에서 유용합니다.

### 1. 기본 개념
`Runnable`은 기본적으로 입력을 받아서 특정 작업을 수행하고, 그 결과를 반환하는 함수형 인터페이스입니다. 이는 다양한 형태의 작업을 추상화하여 일관된 방식으로 처리할 수 있게 해줍니다. 예를 들어, 텍스트 전처리, 모델 호출, 후처리 등을 `Runnable`로 구현할 수 있습니다.

### 2. 구성 요소
`Runnable`은 다음과 같은 주요 구성 요소로 이루어져 있습니다:

- **입력(Input)**: `Runnable`이 처리할 데이터입니다. 이는 문자열, 숫자, 리스트 등 다양한 형태일 수 있습니다.
- **처리(Processing)**: 입력 데이터를 기반으로 특정 작업을 수행하는 로직입니다. 이 과정에서 AI 모델을 호출하거나, 데이터 변환을 수행할 수 있습니다.
- **출력(Output)**: 처리 결과로 반환되는 데이터입니다. 이는 후속 작업에 사용될 수 있습니다.

### 3. 사용 예시
LangChain에서 `Runnable`을 사용하는 예시는 다음과 같습니다:

```python
from langchain.runnables import Runnable

class MyRunnable(Runnable):
    def run(self, input_data):
        # 입력 데이터 처리
        processed_data = self.process(input_data)
        return processed_data

    def process(self, data):
        # 데이터 전처리 로직
